---
title: Manipulating Data in the Tree
---

In [1]:
# Load data
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

EAD_sample = DATA / 'sample-ead-superior.xml'

# Start ElementTree
import xml.etree.ElementTree as ET

# Load data
tree = ET.parse(EAD_sample)
root = tree.getroot()

# Set the namespace values
ns = {
    'ead' : 'http://ead3.archivists.org/schema/'
}
ET.register_namespace('', 'http://ead3.archivists.org/schema/')

## Manipulating XML Data with ElementTree

The `etree` module can also be used to modify XML, including adding/modifying/removing attributes, reading and modifying elements, or adding text within an element.

### Functions to Review, Modify, or Add Data

- `.set()` - allows you to add ("set") specified attributes
- to remove attributes, use `.del()` - this works because the ElementTree processes attributes as a dictionary
- `.append()` - to add a new Element or "tag" to an Element. If input as a string, this will use `.fromstring()` (if providing an element whole cloth written out with tags, attributes, and text) or `.text` (if only providing the wrappend contents)
- `.insert()` - wimilar to the above, this can be used to insert an Element in a specific location

#### Modifying attributes with `.set()` and `.del()`

To create new attributes, etree provides the `.set()` function. This takes the name of the desired attribute and the value as strings; it is called on an Element object.

A potentially useful instance is to add missing values or update invalid values. For example, the EAD `control` element
allows a `@language` attribute.

In [2]:
control = tree.find('ead:control', namespaces=ns)
control.attrib

{'countryencoding': 'iso3166-1',
 'dateencoding': 'iso8601',
 'langencoding': 'iso639-2b'}

In the current example, the ISO langauge code is provided, but the language name is not given. Using this data would require looking up the language name in the ISO list. For ease of reference, it is worth adding the language descriptor as well as the language code. The `.set()` function is one way to do this.

In [3]:
control.set('language', 'en-US')

The `control` element now contains a `@language` attribute. This can be verified by listing the attributes:

In [4]:
control.attrib

{'countryencoding': 'iso3166-1',
 'dateencoding': 'iso8601',
 'langencoding': 'iso639-2b',
 'language': 'en-US'}

#### Adding an element with `.append()`

In other cases, you may want to add new elements. For this, the `.append()` or `.insert()` functions are most useful.

Let's try adding a `subtitle` element to `titlestmt`, since it currently only includes the `titleproper` element. One way to do this is to append that subtitle to the existing `titlestmt` element.

In [5]:
#| label: ex-element-construct
# isolate or identify the target element
titlestmt = control.find('ead:filedesc/ead:titlestmt', namespaces=ns)

# common pattern to construct an element
subtitle = ET.Element('{http://ead3.archivists.org/schema/}subtitle')
subtitle.text = 'Descending to the Depths of the Great Lakes'
titlestmt.append(subtitle)

Above, the first line creates a variable that refers to the `titlestmt` element using a find function. Note the use of the namespaces dictionary to allow for naming elements with the `ead:` prefix.

Then, use a Element "constructor" technique, like the pattern
illustrated in the last three lines of [](#ex-element-construct).
In that pattern, etree's `.Element()` method is assigned to a variable named `subtitle`; the full qualified name is used, with the EAD URI prepended to the tag.
Then, the string "Descending to the Depths of the Great Lakes" is assigned to the new element's `.text` property.
Finally, the new subtitle data is provided to an `.append()` function applied to the `titlestmt` variable.

To review the changes, use the `.tostring` method previously introduced to display the updated `control` element as follows. The `control` variable, which contains the EAD element already has the namesapce information, so there is no need to add that again. The `tostring` function requires the `encoding` argument to be set to `unicode`, and for readability select a blank value for the `default_namespace` argument.

In [6]:
print(ET.tostring(control, encoding='unicode', default_namespace=''))

<control xmlns="http://ead3.archivists.org/schema/" countryencoding="iso3166-1" dateencoding="iso8601" langencoding="iso639-2b" language="en-US">
        <recordid instanceurl="http://jajohnst.si.umich.edu/fake-ead.xml">1234</recordid>
        <filedesc>
            <titlestmt>
                <titleproper>A Finding Aid for the Superior Papers</titleproper>
            <subtitle>Descending to the Depths of the Great Lakes</subtitle></titlestmt>
            <publicationstmt>
                <publisher>University of Michigan School of Information</publisher>
                <date normal="2022-09-01">September 2022</date>
            </publicationstmt>
        </filedesc>
    </control>
    


:::{exercise} Appending or Inserting an element as a string
:label: ex-append-fromstring
Another way to appraoch the "insert" task is to provide a fully formed XML fragement. For example, if you were working in an editor, you could just type in the string `<subtitle>Descending to the Depths of the Great Lakes</subtitle>`.
The `.fromstring` function appears to make this possible.
Does this approach work? What challenges arise if you attempt this?
:::

:::{solution} ex-append-fromstring
The following code is one way to test this:

```{code} python
titlestmt = control.find('ead:filedesc/ead:titlestmt', namespaces=ns)

# insert the element as a valid element string
titlestmt.append(ET.fromstring('<subtitle>Descending to the Depths of the Great Lakes</subtitle>'))
```

If you use the above method, a few challenges may arise, such as:

- The inserted element lacks a namespace. If you later output the tree, python will not recognize this as part of the EAD namespace and add information that makes the element look odd when serialized.
- The new element may be more difficult to process as EAD
:::

#### Removing data with `del()`

To remove attributes, use the `del()` function which is a standard dictionary operation since Python treats the attributes of any Element object as a dictionary.

Let's say that now the `@language` attribute is present,
the `@langencoding` attribute is not needed. It may be removed as follows.

In [7]:
del(control.attrib['langencoding'])

control.attrib

{'countryencoding': 'iso3166-1',
 'dateencoding': 'iso8601',
 'language': 'en-US'}

The resulting `control` should now appear like this when printed as text: 

In [8]:
print(ET.tostring(control, encoding='unicode', default_namespace=''))

<control xmlns="http://ead3.archivists.org/schema/" countryencoding="iso3166-1" dateencoding="iso8601" language="en-US">
        <recordid instanceurl="http://jajohnst.si.umich.edu/fake-ead.xml">1234</recordid>
        <filedesc>
            <titlestmt>
                <titleproper>A Finding Aid for the Superior Papers</titleproper>
            <subtitle>Descending to the Depths of the Great Lakes</subtitle></titlestmt>
            <publicationstmt>
                <publisher>University of Michigan School of Information</publisher>
                <date normal="2022-09-01">September 2022</date>
            </publicationstmt>
        </filedesc>
    </control>
    


:::{exercise}
:label: ex-input-output-diffs
You may notice some slight differences between the output and the original XML that was imported. Do these seem significant? Why or why not? 
:::

:::{solution} ex-input-output-diffs
:label: sol-input-output-diffs
:class: dropdown
**TODO: review this section; some differences changed with the new example**

When you run the above two cells, you will notice that the `xml-model` declaration disappears. It is replaced by a different namespace declaration contined in the root tag (in this case, an `xmlns:ead` attribute). The elements are transcribed slightly differently as well: instead of bare EAD elements (like `ead` and `control`), these are now prefixed (thus `ead:ead` or `ead:control`). While this makes the file slightly longer, this is standard and also creates well-formed and valid XML. In fact, many machine-generated XML files will use this sort of prefixed convention. 
:::

In [ ]:
## examples of modify above

## Writing XML with etree

To write an XML file, use the `.write()` function. When applied to an ElementTree object, this will write a valid XML
out to the filename passed as an argument.
Using the `tree` / `root` variable parsing pattern used above, applying the `.write()` to the `tree` variable will write out the XML. 

A few additional arguments must be passed to `.write()` in order to create valid XML. First, you will need to register any namespaces that are used in the document. To register namespaces, use the `.register_namespace()` function. This can be used multiple times to register multiple namespaces within a single script.

To establish a primary namespace (i.e., one that is assumed for the whole document, supplied in an `xmlns` attribute on the root element, and not prepended to each element), use a blank reference in the first argument passed to the `register_namespace()` function. Below, the `register_namespace()` function is called multiple times to register EAD, MODS, DublinCore, and the basic W3C schema for XML (many shared attributes among these schemes is inherited from the W3C XML schema). (To learn more about these metadata structure standards, follow the associated links.) 

Depending on how you register namespaces, your XML document may look slightly different: if you do not establish a primary namespace each tag will be prepended with the namespace. That can look a bit redundant, but it is specific and still valid XML. As the [MODS User Guide states](https://www.loc.gov/standards/mods/userguide/introduction.html), "Within a record or group of records it is optional to use the "mods" prefix before each element (and before the "mods" namespace declaration), since the MODS namespace is indicated in the record. It is most useful to use the prefix "mods:" before each element when combining a MODS record with XML data from another namespace." (referenced October 2022) 

In [ ]:
# to establish an unprefixed namespace, use a blank in the first argument:
ET.register_namespace('', 'http://ead3.archivists.org/schema/')
# alternatively, specify the 'ead' prefix to be extra specific
#ET.register_namespace('ead', 'http://ead3.archivists.org/schema/')
ET.register_namespace('mods', 'http://www.loc.gov/mods/v3')
ET.register_namespace('dc', 'http://purl.org/dc/elements/1.1/')
ET.register_namespace('xsi', 'http://www.w3.org/2001/XMLSchema')

In addition, you must specify how you want eTree to write the file. To do this, you will provide a file name for output, as well as specify the following variables: 

* `xml_declaration` variable - takes a boolean (True or False) 
* `encoding` variable - to specify the character encoding (here use 'utf-8'), and 
* `method` variable to specify for the writer to use (the default here is `xml`, but you can also request `xhtml` or `html`) provided as a string.

Notice that the first two variables you need are the same information provided in a standard XML document declaration statement:

```xml
<?xml version='1.0' encoding='utf-8'?>
```

A full `write` function might look like this:

In [ ]:
tree.write(ead_file, xml_declaration=True, encoding='utf-8', method='xml')

## Summary

This section illustrates techniques for modifying and reviewing parts of an XML tree structure. These are useful if you need to add data, change it, or identify invalid entries. The section also summarizes how to print out valid and well formed XML that includes namespaces, XML declarations, and other important structural elements.